In [ ]:
# Esame 654AA - a.a. 2025/2026
# Studenti: Leonardo Celati, Samuele Taviano
# Matricole: 660185, ?

<h1>CUP Datasets SVM</h1>
<p>Exploring the cup dataset with SVM Classifier.</p>
<hr/>

In [ ]:
import importlib

import numpy as np
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, mean_absolute_error, mean_squared_error, \
    r2_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, cross_val_predict, KFold
import cup_common as cc
import svm_common as sc

In [ ]:
importlib.reload(sc)
importlib.reload(cc)

In [ ]:
# Note: The regularization parameter C is explored on a logarithmic scale over four orders of magnitude,
# from strong (0.1) to weak regularization (100), following standard practice for SVM hyperparameter tuning.
param_grid_default = {
    "svr__estimator__kernel": ["rbf", "linear", "poly", "sigmoid"],
    "svr__estimator__C": [0.1, 1, 10, 100],
    "svr__estimator__gamma": ["scale", 0.1, 0.01, 0.001],
    "svr__estimator__epsilon": [0.01, 0.05, 0.1, 0.2],
    "svr__estimator__degree": [2, 3, 4],
}

# We only test SVC
pipe_default = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("svr", MultiOutputRegressor(SVR()))
])

scoring="neg_mean_squared_error"

# CV params
n_split = 5
cv_default = KFold(n_splits=n_split, shuffle=True, random_state=42)


<h4>Data Loading</h4>
<p>Load and prepare data</p>

In [ ]:
df_train, df_test = cc.load_set()
X_tr, y_tr = cc.prepare_dataset(df_train)
X_ts, y_ts = cc.prepare_dataset(df_test)
features_names = X_tr.columns
cc.dataset_introspection(df_train, df_test)

<h4>Model Selection</h4>
<p>Finding the best model with GridSearchCV.</p>

In [ ]:
# Perform a grid search
grid = GridSearchCV(
    estimator = pipe_default,
    param_grid = param_grid_default,
    cv = cv_default,
    scoring = scoring,
    n_jobs = -1 # use all available cores
)

# Fit TR features and labels into grid
grid.fit(X_tr, y_tr)
best_model = grid.best_estimator_
sc.grid_introspection(grid)

<h4>Model Assesment</h4>
<p>Collecting various metrics for model assesment.</p>

<h5>Cross-validated prediction</h5>
<p>Cross-validated predictions are computed using a k-fold cross-validation strategy, where the dataset is partitioned into multiple folds and, at each iteration, the model is trained on the training folds and used to predict the held-out fold.
This procedure ensures that each prediction is generated by a model that has not been trained on the corresponding sample, providing an unbiased estimate of the model’s generalization behavior on unseen data.</p>

In [ ]:
importlib.reload(cc)
kfold_result = sc.run_kfold(best_model, np.asarray(X_tr), y_tr, cv_default)
cc.mee_table(kfold_result, "SVM")

In [ ]:
importlib.reload(cc)
cc.plot_kfold_bar_vl_mee(kfold_result["hist_vl_mee"])

In [ ]:
cc.plot_kfold_bar_vl_rmse(kfold_result["hist_vl_rmse"])

<h5>Learning Curves</h5>
<p>This plot shows the evolution of training and cross-validation errors as the training set size increases. The learning curve helps assess whether the model suffers from underfitting or overfitting and whether performance is limited by model capacity or by intrinsic noise in the data.</p>

In [ ]:
sc.plot_learning_curve_svr(best_model, X_tr, y_tr, cv_default, scoring=grid.scoring)

<h4>Test predictions</h4>
<p>Test predictions on the test set.</p>

In [ ]:
y_pred = best_model.predict(X_ts)

In [ ]:
print(y_pred)

<hr/>